# Three injection points, one energy, six geometries

**600 events at exactly 10 TeV, placed at three fixed vertices, replayed through every geometry.** Energy, vertex and the injected events themselves are held fixed; only the detector changes. Any difference between arms *within* an injection point is geometry and nothing else.

1. **Geometry survey** — runs with no simulation at all, straight from the Prometheus geofiles.
2. **Pairing check** — verifies arm A's event *i* really is arm B's event *i*, and that the vertices landed on the three points.
3. **Reconstruction** — deliberately simple, geometry-free estimators, broken down by injection point.

Part 1 runs immediately. Parts 2–5 need a run from
`python -m src.prometheus_simulation.simulate --out runs/pilot --execute`.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from prometheus_simulation import geometry as geo
from prometheus_simulation.physics import PhysicsParameters

# Categorical palette: muted teal / orange / violet / blue / raspberry / olive.
# Validated against the light surface (lightness band, chroma floor,
# normal-vision separation and contrast all pass; the olive-raspberry pair sits
# in the CVD floor band, so every figure also carries a legend and markers).
PALETTE = ["#008B7A", "#C2410C", "#6D28A8", "#2E6FB0", "#A8325A", "#4D7C0F"]
INK, INK2, MUTED, GRID = "#1c1c1a", "#4a4a46", "#7a7a74", "#e4e4df"
SURFACE = "#fcfcfb"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "semibold",
    "legend.frameon": False, "figure.dpi": 120,
    "lines.linewidth": 2, "lines.markersize": 5,
})

POINT_STYLE = {"centre": "-", "radial": "--", "vertical": ":"}


def show(df, fmt=None):
    """Format when jinja2 is available, fall back to a plain frame otherwise."""
    try:
        s = df.style.format(fmt) if fmt else df.style
        _ = s.to_html()
        return s
    except Exception:
        return df.round(4) if fmt else df


GEODIR = REPO / "src" / "prometheus_simulation" / "external" / "prometheus" / "resources" / "geofiles"
RUN = REPO / "runs" / "pilot"        # <-- point this at your run
params = PhysicsParameters.from_yaml(
    REPO / "src" / "prometheus_simulation" / "config" / "physics_default.yaml")
print("geofiles:", GEODIR.exists(), "| run:", RUN.exists())
print(f"design: {params.n_events} events = {params.events_per_point} x "
      f"{len(params.injection_points)} points at {params.energy_gev:,.0f} GeV")

## Part 1 — the geometries

Parsed from the geofiles Prometheus ships. The string counts are the check: 150 / 115 / 1211 / 3 / 8 / 86 match NuBench Table 1 exactly.

In [ ]:
geoms = geo.load_geometries(GEODIR)
COLOR = {k: PALETTE[i % len(PALETTE)] for i, k in enumerate(geoms)}
MARKER = dict(zip(geoms, ["o", "s", "^", "D", "v", "P"]))

survey = pd.DataFrame([g.as_row() for g in geoms.values()], index=list(geoms))
survey["geofile"] = [geo.NUBENCH_GEOFILES[k] for k in geoms]
survey["medium_nubench"] = [geo.NUBENCH_MEDIA[k] for k in geoms]
show(survey[["geofile", "n_modules", "n_strings", "offset_z", "r_horizontal_m",
             "half_height_m", "depth_m", "medium_header", "medium_nubench"]])

### Footprints, with the three injection points

Each panel is on its own scale — the six span 34× in radius. The dashed circle is the **common region**: radius ≤ 57.7 m, set by `triangle`. Every injection point must sit inside it, in every detector.

In [ ]:
region = geo.common_region(geoms)
PTS = params.injection_points
Rc = region["max_radius_m"]

fig, axes = plt.subplots(2, 3, figsize=(11, 7.4))
for ax, (k, g) in zip(axes.ravel(), geoms.items()):
    xy = g.coords[:, :2] - g.offset[:2]
    size = float(np.clip(900 / np.sqrt(max(g.n_modules, 1)), 4, 40))
    ax.scatter(xy[:, 0], xy[:, 1], s=size, color=COLOR[k], alpha=0.8, edgecolors="none")
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(Rc * np.cos(th), Rc * np.sin(th), ls="--", lw=1.4, color=INK2)
    # `centre` and `vertical` share (x, y) -- they differ only in z, which this
    # projection cannot show. Label the group once rather than overprinting.
    groups = {}
    for name, (px, py, pz) in PTS.items():
        groups.setdefault((px, py), []).append(name)
    for (px, py), names in groups.items():
        ax.scatter([px], [py], marker="*", s=190, color=INK,
                   edgecolors=SURFACE, linewidths=1.2, zorder=5)
        left = px <= 0
        ax.annotate(" + ".join(names), (px, py), textcoords="offset points",
                    xytext=(-9, -16) if left else (9, 8), fontsize=8, color=INK,
                    ha="right" if left else "left")
    lim = g.r_horizontal * 1.12
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_title(f"{k}   {g.n_strings} strings · {g.n_modules} OMs\nr = {g.r_horizontal:.0f} m",
                 fontsize=9)
    ax.set_xlabel("x − x₀  [m]"); ax.set_ylabel("y − y₀  [m]")
fig.suptitle("Detector footprints (own scale each) — ★ injection points, dashed: common region (r = %.0f m)" % Rc,
             y=1.005, fontsize=11, color=INK)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for k, g in geoms.items():
    ax.scatter(g.r_horizontal, g.half_height * 2, s=90, color=COLOR[k],
               marker=MARKER[k], edgecolors=SURFACE, linewidths=1.5, label=k, zorder=3)
    ax.annotate(k, (g.r_horizontal, g.half_height * 2), textcoords="offset points",
                xytext=(11, -3), fontsize=9, color=INK2, ha="left", va="center")
ax.axvline(Rc, ls="--", lw=1.4, color=INK2)
ax.set_ylim(bottom=0)
ax.annotate("common region\nradius", (Rc, ax.get_ylim()[0]), textcoords="offset points",
            xytext=(8, 14), fontsize=9, color=INK2, ha="left", va="bottom")
ax.set_xscale("log"); ax.set_xlabel("horizontal radius  [m]")
ax.set_ylabel("instrumented height  [m]")
ax.set_title("Instrumented scale — the six geometries span 34× in radius")
ax.legend(ncol=2, fontsize=9, loc="lower right", bbox_to_anchor=(0.99, 0.06))
fig.tight_layout()

### Why fixed points and not a sampled volume

A shared injection cylinder has to serve a 58 m detector and a 1950 m one at once. The sweep below is the evidence that no radius works: at every choice, either the small geometries see almost none of the injected events (`worst_detector_sees`) or the large ones are probed in a fraction of a percent of their volume (`worst_probes_detector`). The two never rise together.

Fixed points sidestep it — the constraint becomes a containment check, not a trade-off.

In [ ]:
print(f"common region: radius ≤ {region['max_radius_m']:.1f} m "
      f"(set by {region['binding_radius']}), |z| ≤ {region['max_abs_z_m']:.1f} m "
      f"(set by {region['binding_height']})\n")
display(show(pd.DataFrame(geo.check_points(PTS, geoms)).set_index("point"),
             {"radius_m": "{:.1f}", "abs_z_m": "{:.1f}",
              "radius_headroom": "{:.2f}", "z_headroom": "{:.2f}"}))
sweep = pd.DataFrame(geo.sweep_cylinders(geoms))

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(sweep.radius_m, sweep.worst_detector_sees, marker="o", color=PALETTE[0],
        label="worst detector_sees  (events a detector can contain)")
ax.plot(sweep.radius_m, sweep.worst_probes_detector, marker="s", color=PALETTE[1],
        label="worst probes_detector  (volume the set explores)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("injection cylinder radius  [m]"); ax.set_ylabel("worst case over the six geometries")
ax.set_title("No balanced cylinder exists — the two curves never rise together")
ax.legend(fontsize=9, loc="lower center")
fig.tight_layout()

## Part 2 — load the event set and check the pairing

Two acceptance gates. `check_pairing` must show `paired = True` for **every** arm, and `vertex_residual_max_m` must be ~0 — the vertices really landed on the three points. If either fails, stop; nothing below is interpretable.

In [ ]:
from prometheus_simulation.readout import load_event_set, check_pairing
from prometheus_simulation import recon

HAVE_RUN = (RUN / "plan.json").exists()
if not HAVE_RUN:
    print(f"No run at {RUN}. Generate one with:\n"
          f"  python -m src.prometheus_simulation.simulate --out runs/pilot --execute\n"
          f"Part 1 above is complete without it.")
    truth = hits = plan = None
else:
    truth, hits, plan = load_event_set(RUN)
    arms = [a["arm"] for a in plan["arms"] if a["role"] == "geometry"]
    print(f"{truth.arm.nunique()} arms · {len(truth)} event-rows · {len(hits)} hits")
    print("reference geometry:", plan["reference_geometry"])
    print("vertex residual (m):", plan.get("vertex_residual_max_m", "n/a"))
    print("events per point:", truth[truth.arm == arms[0]].point.value_counts().to_dict())

In [ ]:
if HAVE_RUN:
    pair_table = check_pairing(truth, plan)
    display(show(pair_table))
    assert pair_table.paired.all(), "PAIRING BROKEN — do not interpret anything below"
    assert plan.get("vertex_residual_max_m", 1) < 1e-6, "VERTICES NOT ON THE POINTS"
    print("pairing OK, vertices on target")

## Part 3 — one event, every geometry

The same 10 TeV interaction as each detector records it. Same vertex, same direction, same energy — only the instrument changes.

In [ ]:
if HAVE_RUN:
    cand = truth[(truth.point == "centre") & (truth.n_hits >= 4)]
    eid = int(cand.groupby("event_id").size().sort_values(ascending=False).index[0])

    fig, axes = plt.subplots(2, 3, figsize=(11.5, 7))
    for ax, arm in zip(axes.ravel(), arms):
        h = hits[(hits.arm == arm) & (hits.event_id == eid)]
        g = geoms[arm]
        mods = g.coords - g.offset
        ax.scatter(mods[:, 0], mods[:, 2], s=1.5, color=GRID, edgecolors="none", zorder=1)
        if len(h):
            sc = ax.scatter(h.x - g.offset[0], h.z - g.offset[2], c=h.t, cmap="viridis",
                            s=26, edgecolors=SURFACE, linewidths=0.5, zorder=3)
            plt.colorbar(sc, ax=ax, label="hit time [ns]", fraction=0.046)
        row = truth[(truth.arm == arm) & (truth.event_id == eid)].iloc[0]
        ax.scatter([row.vertex_x - g.offset[0]], [row.vertex_z - g.offset[2]],
                   marker="*", s=200, color=INK, edgecolors=SURFACE, linewidths=1.2, zorder=6)
        ax.set_title(f"{arm} — {int(row.n_hits)} hits", fontsize=9)
        ax.set_xlabel("x − x₀ [m]"); ax.set_ylabel("z − z₀ [m]"); ax.set_aspect("equal")
    t0 = truth[(truth.arm == arms[0]) & (truth.event_id == eid)].iloc[0]
    fig.suptitle(f"Event {eid} at the `centre` point — {t0.energy_gev:,.0f} GeV, "
                 f"zenith {np.degrees(t0.zenith_rad):.0f}°  ·  ★ true vertex",
                 y=1.01, fontsize=11, color=INK)
    fig.tight_layout()

## Part 4 — reconstruction, by injection point

Three estimators, none trained, none geometry-aware:

- **vertex** — centroid of the 5 earliest hits
- **direction** — principal axis of the hit cloud, signed by time ordering
- **light yield** — hit multiplicity. With energy fixed there is no resolution to fit; the observable *is* the yield, and how it changes between the three vertices measures how position-dependent each geometry's response is.

In [ ]:
if HAVE_RUN:
    df = recon.reconstruct(truth, hits)
    print("── per geometry ──")
    display(show(recon.summarise(df), {
        "trigger_efficiency": "{:.3f}", "median_vertex_error_m": "{:.1f}",
        "median_angular_error_deg": "{:.1f}", "median_elongation": "{:.3f}"}))
    print("── per geometry × injection point ──")
    display(show(recon.summarise(df, by=["arm", "point"]), {
        "trigger_efficiency": "{:.3f}", "median_vertex_error_m": "{:.1f}",
        "median_angular_error_deg": "{:.1f}", "median_elongation": "{:.3f}"}))

In [ ]:
if HAVE_RUN:
    pts = list(PTS)
    fig, axes = plt.subplots(2, len(pts), figsize=(4 * len(pts), 7), sharex="row")
    for j, pt in enumerate(pts):
        for i, (col, label, logx) in enumerate(
                [("vertex_error_m_early", "vertex error [m]", True),
                 ("angular_error_deg", "angular error [deg]", False)]):
            ax = axes[i, j]
            for k in arms:
                sel = df[(df.index.get_level_values("arm") == k) & (df["point"] == pt)]
                v = sel[col].dropna()
                v = v[v > 0] if logx else v
                if len(v) < 5:
                    continue
                bins = np.logspace(np.log10(v.min()), np.log10(v.max()), 24) if logx \
                    else np.linspace(0, 180, 24)
                ax.hist(v, bins=bins, histtype="step", lw=2, color=COLOR[k], label=k)
            if logx:
                ax.set_xscale("log")
            ax.set_xlabel(label); ax.set_ylabel("events")
            if i == 0:
                ax.set_title(f"point: {pt}   {PTS[pt]}", fontsize=10)
    axes[0, 0].legend(fontsize=8, ncol=2)
    fig.suptitle("Same events, same estimator — geometry across columns of fixed vertex",
                 y=1.01, fontsize=11, color=INK)
    fig.tight_layout()

In [ ]:
if HAVE_RUN:
    ly = recon.light_yield(df)
    piv = ly.pivot(index="arm", columns="point", values="median").reindex(arms)
    lo = ly.pivot(index="arm", columns="point", values="q16").reindex(arms)
    hi = ly.pivot(index="arm", columns="point", values="q84").reindex(arms)

    x = np.arange(len(piv)); w = 0.8 / len(pts)
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    for j, pt in enumerate(pts):
        off = (j - (len(pts) - 1) / 2) * w
        vals = piv[pt].to_numpy(dtype=float)
        err = np.abs(np.vstack([vals - lo[pt].to_numpy(dtype=float),
                                hi[pt].to_numpy(dtype=float) - vals]))
        ax.bar(x + off, vals, width=w * 0.9, color=PALETTE[j], edgecolor=SURFACE,
               linewidth=2, label=f"{pt} {PTS[pt]}")
        ax.errorbar(x + off, vals, yerr=err, fmt="none", ecolor=INK2, elinewidth=1.2, capsize=2)
    ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(piv.index, rotation=15)
    ax.set_ylabel("median hits per event  (16–84%)")
    ax.set_title(f"Light yield at {params.energy_gev:,.0f} GeV — energy fixed, only the vertex and the geometry change")
    ax.legend(fontsize=9)
    fig.tight_layout()

    spread = ly.groupby("arm")["point_spread_ratio"].first().reindex(arms)
    print("Position dependence (max/min median yield across the three points):")
    for k, v in spread.items():
        print(f"  {k:12s} {v:6.2f}×")

## Part 5 — is the geometry effect above the photon-stochasticity floor?

Photon propagation reruns per arm and is seedable only to Poisson level, so the *same* event in the *same* detector already gives different light. The `photon_null` arm measures that floor. **A cross-geometry difference only means something as a multiple of it.** An AUC comfortably above 0.75 says the geometry effect is real at this statistics; near 0.5 says it is not separated.

In [ ]:
if HAVE_RUN:
    null_arm = next((a["arm"] for a in plan["arms"] if a["role"] == "photon_null"), None)
    ref = plan["reference_geometry"]
    if null_arm is None or null_arm not in df.index.get_level_values("arm"):
        print("no photon_null arm in this run")
    else:
        base = df.xs(ref, level="arm")["n_hits"]
        null = df.xs(null_arm, level="arm")["n_hits"]
        c = base.index.intersection(null.index)
        d_null = (np.abs(null.loc[c] - base.loc[c]) / np.maximum(base.loc[c], 1)).to_numpy()

        rows = []
        for k in arms:
            if k == ref:
                continue
            other = df.xs(k, level="arm")["n_hits"]
            c2 = base.index.intersection(other.index)
            a = (np.abs(other.loc[c2] - base.loc[c2]) / np.maximum(base.loc[c2], 1)).to_numpy()
            rows.append({"arm": k, "median_cross": float(np.median(a)),
                         "null_p95": float(np.percentile(d_null, 95)),
                         "ratio_to_null_p95": float(np.median(a) / np.percentile(d_null, 95)),
                         "auc_vs_null": float((a[:, None] > d_null[None, :]).mean())})
        effect = pd.DataFrame(rows).set_index("arm")
        effect["separated"] = effect.auc_vs_null > 0.75
        display(show(effect, {"median_cross": "{:.3f}", "null_p95": "{:.3f}",
                              "ratio_to_null_p95": "{:.2f}", "auc_vs_null": "{:.3f}"}))

        fig, ax = plt.subplots(figsize=(7.5, 4))
        y = np.arange(len(effect))
        ax.barh(y, effect.auc_vs_null, height=0.55,
                color=[COLOR[k] for k in effect.index], edgecolor=SURFACE, linewidth=2)
        ax.axvline(0.75, ls="--", lw=1.6, color=INK2)
        ax.annotate("separation threshold", (0.75, len(effect) - 0.4),
                    textcoords="offset points", xytext=(6, 0), fontsize=9, color=INK2)
        ax.set_yticks(y); ax.set_yticklabels(effect.index)
        ax.set_xlim(0.4, 1.0)
        ax.set_xlabel(f"AUC: geometry change vs photon seed alone (reference = {ref})")
        ax.set_title("Is the geometry effect above the photon-stochasticity floor?")
        for yi, v in zip(y, effect.auc_vs_null):
            ax.annotate(f"{v:.2f}", (v, yi), textcoords="offset points",
                        xytext=(6, -3), fontsize=9, color=INK2)
        fig.tight_layout()

### Recorded physics

Everything the run was generated under, with provenance tags. The `ask` list is what NuBench never published — our values there are declared deviations, not reproductions.

In [ ]:
rec_path = RUN / "physics_record.json"
if rec_path.exists():
    rec = json.loads(rec_path.read_text())
    print("fingerprint:", rec["fingerprint"])
    print("prometheus :", rec["environment"]["prometheus_commit"])
    print("repo       :", rec["environment"]["repo_commit"])
    print("\nDECLARED DEVIATIONS (unpublished in NuBench):")
    for k in rec["unresolved_parameters"]:
        print(f"  {k:22s} = {rec['physics'][k]}")
else:
    print("defaults; fingerprint:", params.fingerprint())
    print("would be declared deviations:", params.unresolved())